In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("/kaggle/input/datasets/sanjananpatel/merged-all-datasets/merged_all_sheets.csv")

df.columns = df.columns.str.strip()
df.columns = df.columns.str.strip().str.replace(' ', '_')

df = df.replace([np.inf, -np.inf], np.nan)
df = df.fillna(0)

In [2]:
counts = df["Label"].value_counts()

valid = counts[counts >= 100].index

df = df[df["Label"].isin(valid)]

In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Label"] = le.fit_transform(df["Label"])

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop("Label", axis=1)
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [5]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [6]:
lgb = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    num_leaves=64,
    class_weight="balanced",
    n_jobs=-1,
    verbose=-1
)

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=10,
    learning_rate=0.05,
    tree_method="hist",
    eval_metric="mlogloss",
    n_jobs=-1
)

cat = CatBoostClassifier(
    iterations=400,
    learning_rate=0.05,
    depth=8,
    verbose=0
)

In [7]:
from sklearn.ensemble import VotingClassifier

model = VotingClassifier(
    estimators=[
        ("lgb", lgb),
        ("xgb", xgb),
        ("cat", cat)
    ],
    voting="soft",
    n_jobs=-1
)

In [8]:
model.fit(X_train, y_train)

VotingClassifier(estimators=[('lgb',
                              LGBMClassifier(class_weight='balanced',
                                             learning_rate=0.05,
                                             n_estimators=400, n_jobs=-1,
                                             num_leaves=64, verbose=-1)),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_m...
                                            learning_rate=0.05, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=10,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=400, n_jobs=-1,
                                            num_parallel_tree=None, ...)),
                             ('cat',
                              <catboost.core.CatBoostClassifier object at 0x7c3026f4a540>)],
                 n_jobs=-1, voting='soft')

In [9]:
from sklearn.metrics import classification_report

pred = model.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    454619
           1       0.79      0.91      0.84       393
           2       1.00      1.00      1.00     25606
           3       1.00      1.00      1.00      2059
           4       1.00      1.00      1.00     46215
           5       0.99      0.99      0.99      1100
           6       0.99      1.00      1.00      1159
           7       1.00      1.00      1.00      1588
           8       0.99      1.00      1.00     31786
           9       1.00      1.00      1.00      1179
          10       0.77      0.88      0.82       301
          11       0.57      0.38      0.46       130

    accuracy                           1.00    566135
   macro avg       0.93      0.93      0.93    566135
weighted avg       1.00      1.00      1.00    566135



In [10]:
import joblib

joblib.dump(model, "nids_model.pkl")

['nids_model.pkl']